# Практическое задание 2. Мошенничество по банковским картам

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса

Магистратура «Бизнес-аналитика и цифровые решения в финансах»

---

Служба безопасности банка каждый месяц вручную разбирает 2 % операций по
картам — те, что модель считает самыми подозрительными. Нужно оценить,
насколько подозрительна каждая операция декабря 2019 года, так чтобы разбор
нашёл как можно больше мошенничеств. Отметки о мошенничестве есть
в истории; отметку любой операции ноября можно получить, заказав её
проверку у бота курса, но проверки стоят денег.

У каждого свой вариант: операции у всех одни, а отметки о мошенничестве —
свои. Номер варианта смотрите в файле
`распределение_вариантов_по_студентам.xlsx` в нашем чате.

Что требуется, как считается итог и как ставится оценка — в тексте
задания: https://github.com/Rimest/ml_ai_course_gsb_hse/blob/main/homeworks/hw2_fraud/README.md

**Что сдать:** этот ноутбук `ФАМИЛИЯ_ИО.ipynb` с выполненными ячейками и файл
`ФАМИЛИЯ_ИО_predictions.csv` с оценками.

## Преамбула

Впишите фамилию с инициалами и номер варианта и выполните ячейку. Сид `SEED`
подставляйте везде, где есть параметр `random_state`: ноутбук должен при
повторном запуске сверху вниз давать те же числа.

In [ ]:
## ── воспроизводимость ──────────────────────────────────────────────
import random

import numpy as np

STUDENT_NAME = <ваша фамилия и инициалы>  # например, "ИВАНОВ_ИИ"
SEED = <номер вашего варианта>
VARIANT = <номер вашего варианта>
random.seed(SEED)      # стандартный модуль random
np.random.seed(SEED)   # numpy

# Своего генератора у pandas и scikit-learn нет — оба берут случайность
# из numpy, так что строка выше покрывает и их. Но там, где параметр
# random_state есть, передавайте SEED явно.

## Данные

Данные лежат на Яндекс Диске: https://disk.yandex.ru/d/tetGliBdgbgjUA

Скачайте оттуда четыре файла и положите рядом с ноутбуком (в Colab —
загрузите в папку сессии). Ячейка ниже читает их в четыре таблицы.

| Таблица | Файл | Что внутри |
|---|---|---|
| `history` | `transactions_history.parquet` | операции января 2017 — октября 2019 |
| `pool` | `transactions_pool.parquet` | операции ноября 2019, отметки — только через проверки |
| `test` | `transactions_test.parquet` | операции декабря 2019, по ним сдаются оценки |
| `labels` | `history_labels_NN.parquet` из папки `labels` | отметки истории вашего варианта: `операция`, `мошенничество` |

`NN` — номер вашего варианта двумя цифрами: `history_labels_07.parquet`,
например. Описание каждого столбца — в тексте задания.

In [ ]:
## ── данные: файлы с Яндекс Диска, положенные рядом с ноутбуком ─────
import pandas as pd

history = pd.read_parquet("transactions_history.parquet")
pool = pd.read_parquet("transactions_pool.parquet")
test = pd.read_parquet("transactions_test.parquet")
labels = pd.read_parquet(f"history_labels_{VARIANT:02d}.parquet")

print(f"история: {len(history):_} операций, отметок: {len(labels):_}".replace("_", " "))
print(f"ноябрь:  {len(pool):_} операций".replace("_", " "))
print(f"декабрь: {len(test):_} операций".replace("_", " "))

## Решение

## Покупка проверок

Проверки заказываются у бота: команда `/check`, затем файл с номерами
операций ноября, затем `/yes`. Бот присылает файл `check_N.csv` с
отметками — положите его рядом с ноутбуком (в Colab — загрузите в папку
сессии).

Ячейка ниже даёт три функции:

| Функция | Что делает |
|---|---|
| `purchase_file(ids, n)` | записывает номера операций ноября в `purchase_n.csv` (`n` — номер покупки: 1, 2, 3…) и проверяет формат: номера из ноября, без повторов, без уже проверенных, не больше 1 000 |
| `check_answers()` | читает все файлы `check_*.csv` рядом с ноутбуком в одну таблицу `операция`, `мошенничество` |
| `cost(n)` | стоимость n проверок за всё задание, $ |

In [ ]:
## ── покупка проверок: файл для бота и ответы бота ──────────────────
import glob


def cost(n):
    "Стоимость n проверок за всё задание, $."
    return 50 * n * (1 + n / 1000)


def dollars(x):
    "12345.6 -> «12 346 $»."
    return f"{x:,.0f} $".replace(",", " ")


def check_answers():
    "Все ответы бота — файлы check_*.csv рядом с ноутбуком — одной таблицей."
    files = sorted(glob.glob("check_*.csv"))
    if not files:
        return pd.DataFrame({"операция": pd.Series(dtype="int64"), "мошенничество": pd.Series(dtype="int64")})
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def purchase_file(ids, n):
    "Записать номера операций ноября в purchase_n.csv; отправьте его боту после команды /check."
    ids = pd.Series(np.asarray(ids)).astype("int64")
    done = check_answers()["операция"]
    problems = []
    if ids.duplicated().any():
        problems.append(f"номера повторяются: {ids.duplicated().sum()}")
    if not ids.isin(pool["операция"]).all():
        problems.append(f"не операции ноября: {(~ids.isin(pool['операция'])).sum()}")
    if ids.isin(done).any():
        problems.append(f"уже проверены: {ids.isin(done).sum()}")
    if len(ids) > 1000:
        problems.append(f"в одной покупке не больше 1 000 операций, а здесь {len(ids)}")
    if problems:
        raise ValueError("; ".join(problems))
    path = f"purchase_{n}.csv"
    pd.DataFrame({"операция": ids}).to_csv(path, index=False, encoding="utf-8")
    total = len(done) + len(ids)
    print(f"{path}: {len(ids)} операций. После покупки проверок станет {total}, "
          f"их стоимость {dollars(cost(total))} (эта покупка: {dollars(cost(total) - cost(len(done)))})")
    return path


print(f"Ответов на проверки рядом с ноутбуком: {len(check_answers())}")

## Файл сдачи

Ячейка ниже собирает `ФАМИЛИЯ_ИО_predictions.csv`. Формат одинаков у всех,
поэтому код готовый — выполните ячейку, менять её не требуется. Кроме полей
преамбулы к этому моменту должны быть две величины:

| Имя | Что это |
|---|---|
| `test` | таблица операций декабря, со столбцом `операция` |
| `score_test` | оценки подозрительности операций `test` в том же порядке: чем больше, тем подозрительнее |

Получится файл с тремя столбцами:

```
вариант,операция,оценка
7,5022410,0.003512
7,5022411,0.418877
```

In [ ]:
## ── файл сдачи ─────────────────────────────────────────────────────
predictions = pd.DataFrame({
    "вариант": VARIANT,
    "операция": test["операция"].values,
    "оценка": np.asarray(score_test, dtype=float),
})
predictions.to_csv(f"{STUDENT_NAME}_predictions.csv", index=False, encoding="utf-8")

## ── проверка перед сдачей ──────────────────────────────────────────
errors = []
if len(predictions) != 146579:
    errors.append(f"в файле {len(predictions)} строк, а в декабре 146 579 операций")
if predictions["операция"].duplicated().any():
    errors.append("одна и та же операция встречается несколько раз")
if not np.isfinite(predictions["оценка"]).all():
    errors.append("оценки должны быть числами без пропусков")
if not isinstance(VARIANT, int) or not 1 <= VARIANT <= 57:
    errors.append("номер варианта должен быть числом от 1 до 57")

n_checks = len(check_answers())
print(f"Строк в файле: {len(predictions)}")
print(f"Проверок куплено (по файлам рядом с ноутбуком): {n_checks}, их стоимость {dollars(cost(n_checks))}")
print("Проверка: " + ("всё в порядке" if not errors else "; ".join(errors)))
predictions.head()

## Как оценивается работа

За работу — до 10 баллов.

| За что | Баллов | Как считается |
|---|---|---|
| Итог | 8 | 8 × (доля разрыва / 0,80) ^ 1,5, с точностью до 0,1; не больше 8 и не меньше 0 |
| Формат сдачи | 1 | `ФАМИЛИЯ_ИО_predictions.csv` прошёл формальную проверку целиком |
| Ноутбук | 1 | приложен |

```
итог         = 500 $ × пойманные мошенничества − стоимость(проверок)
доля разрыва = (итог − итог бейзлайна) / (итог потолка − итог бейзлайна)
```

Пойманные — число мошенничеств среди 2 931 операции декабря
с наибольшей оценкой. Бейзлайн и потолок описаны в тексте задания. Если в
коде ноутбука есть модели не из лекций 5–6, за итог ставится 0.

## Обоснование

Ответьте на вопросы:

1. **Что вы сделали с данными и почему.** Что добавили, преобразовали или
   отбросили и как проверяли, что это помогает.
2. **Какие модели сравнивали.** Какие модели из лекций 5–6 пробовали, как
   проверяли их качество, какие параметры подбирали и что это дало.
   Почему выбрали итоговую модель.
3. **Какие проверки заказали.** Сколько, каких операций и почему именно
   столько.
4. **Какой итог вы ждёте.** Сколько мошенничеств поймает разбор декабря,
   сколько стоили проверки и почему результат может оказаться хуже ожидаемого.
5. **Как вы использовали ИИ.** Какими ИИ-инструментами пользовались
   (чат-ассистенты, подсказки кода в редакторе и другие) и на каких шагах
   работы. Для каждого шага — что именно вы просили, что взяли из ответа,
   что проверили или исправили сами и почему. Если ИИ не использовали,
   так и напишите.